In [1]:
import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\jsteve\OneDrive - Burlington\Documents\Python Scripts\Projects\Data-Science-Projects\medicare-claims-intelligence-agent


In [2]:
from agents.run import run_claims_agent

response = run_claims_agent(
    "What share of positive spending "
    "comes from the top 10%?"
)

response

{'question': 'What share of positive spending comes from the top 10%?',
 'answer': 'The top 10% of the population accounts for **49.3%** of positive spending, according to the synthetic CMS DE-SynPUF data analyzed using the `get_cost_concentration` tool.  \n\n**Supporting source file(s):**  \n- Synthetic CMS DE-SynPUF data analysis results (tool output).  \n\nThis finding reflects the proportion of total positive spending attributed to the highest-spending segment in the synthetic dataset.',
 'tool': 'get_cost_concentration',
 'arguments': {'top_fraction': 0.1},
 'tool_result': {'top_fraction': 0.1,
  'top_percent': 10.0,
  'positive_member_years': 302102,
  'selected_member_years': 30210,
  'total_positive_spending': 1582713170.0,
  'selected_spending': 780230690.0,
  'spending_share_pct': 49.29703655653538},
 'sources': [],
 'data_context': 'CMS DE-SynPUF synthetic Medicare data',
 'synthetic_data_notice': 'Results describe a synthetic analytical population and should not be interpre

In [3]:
# Verifying RAG with the workflow

response = run_claims_agent(
    "How is total annual cost calculated?"
)

print(response["answer"])

print(
    "\nTool:",
    response["tool"],
)

print(
    "Sources:",
    response["sources"],
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

The **total_annual_cost** is a composite analytical measure calculated as the sum of four components:  
**MEDREIMB_IP** (inpatient reimbursement), **MEDREIMB_OP** (outpatient reimbursement), **MEDREIMB_CAR** (carve-out reimbursement), and **rx_total_cost** (total pharmacy cost).  

This definition is explicitly stated in the project’s analytical framework and is intended for comparative analysis within the synthetic CMS DE-SynPUF dataset. It is explicitly noted that this measure does not represent a complete actuarial definition of Medicare spending or total healthcare expenditure.  

**Supporting sources**:  
- limitations.md  
- metric_definitions.md  

(Analytical note: The calculation is specific to the synthetic dataset and does not reflect real-world financial or clinical contexts.)

Tool: knowledge_search
Sources: ['limitations.md', 'metric_definitions.md']


In [4]:
# Verifying segment question

response = run_claims_agent(
    "How is total annual cost calculated?"
)

print(response["answer"])

print(
    "\nTool:",
    response["tool"],
)

print(
    "Sources:",
    response["sources"],
)

The **total_annual_cost** is calculated as a composite analytical measure derived from the sum of four components:  
- `MEDREIMB_IP` (Inpatient Medicare reimbursement)  
- `MEDREIMB_OP` (Outpatient Medicare reimbursement)  
- `MEDREIMB_CAR` (Carrier Medicare reimbursement)  
- `rx_total_cost` (Total pharmacy costs)  

This measure is explicitly defined as an analytical construct for comparative analysis within the project and is not intended as a complete actuarial representation of Medicare spending or healthcare expenditures.  

**Supporting sources**:  
- limitations.md  
- metric_definitions.md  

(Analytical note: The calculation aggregates specific reimbursement fields but excludes proration for cross-year claims, as described in limitations.md.)

Tool: knowledge_search
Sources: ['limitations.md', 'metric_definitions.md']


In [2]:
# checkign the LangGraph workflow

from agents.workflow import run_claims_graph

result = run_claims_graph(
    "What share of positive spending "
    "comes from the top 10%?"
)

print("MODE")
print(result["mode"])

print("\nROUTE")
print(result["route"])

print("\nANSWER")
print(result["answer"])

print("\nSOURCES")
print(result["sources"])

MODE
analytics

ROUTE
tool=<ToolName.COST_CONCENTRATION: 'get_cost_concentration'> arguments={'top_fraction': 0.1}

ANSWER
The top 10% of the population accounts for 49.3% of positive spending, according to the synthetic CMS DE-SynPUF data. This analysis identifies the spending share attributable to the highest-spending segment without implying causation or generalizing to real-world Medicare populations.  

**Supporting source file:** `get_cost_concentration` result with `top_fraction` = 0.1.

SOURCES
[]


In [3]:
result = run_claims_graph(
    "How is total annual cost calculated?"
)

print("MODE")
print(result["mode"])

print("\nROUTE")
print(result["route"])

print("\nANSWER")
print(result["answer"])

print("\nSOURCES")
print(result["sources"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

MODE
knowledge

ROUTE
tool=<ToolName.KNOWLEDGE_SEARCH: 'knowledge_search'> arguments={'query': 'How is total annual cost calculated?'}

ANSWER
The **total_annual_cost** is a composite analytical measure calculated as the sum of four components:  
**MEDREIMB_IP** (inpatient reimbursement), **MEDREIMB_OP** (outpatient reimbursement), **MEDREIMB_CAR** (carrier reimbursement), and **rx_total_cost** (total prescription drug cost).  

This definition is explicitly stated in the synthetic CMS DE-SynPUF data documentation and is intended for analytical comparison within the project. It is explicitly noted that this measure does not represent a complete actuarial definition of Medicare spending or total healthcare expenditure.  

**Supporting sources**:  
- limitations.md#limitations_4  
- metric_definitions.md#metric_definitions_1  

**Note**: This analysis uses synthetic CMS DE-SynPUF data.

SOURCES
['limitations.md', 'metric_definitions.md']


In [2]:
# Test planning agent

from agents.planner import plan_question

plan_question(
    "What share of spending comes from the top 10%?"
)

HybridPlan(mode=<AgentMode.ANALYTICS: 'analytics'>, analytical_question=None, knowledge_query=None)

In [3]:
hybrid_plan = plan_question(
    "Compare persistent high-cost complex members "
    "with pharmacy-intensive chronic members, "
    "explain how each cohort is defined, and summarize "
    "the main utilization differences."
)

hybrid_plan

HybridPlan(mode=<AgentMode.HYBRID: 'hybrid'>, analytical_question='Compare persistent high-cost complex members with pharmacy-intensive chronic members and summarize the main utilization differences.', knowledge_query='Definitions of Persistent High-Cost Complex and Pharmacy-Intensive Chronic members.')

In [5]:
# Checking hybrid branch

from agents.workflow import run_claims_graph

question = (
    "Compare persistent high-cost complex members "
    "with pharmacy-intensive chronic members, "
    "explain how each cohort is defined, and summarize "
    "the main utilization differences."
)

result = run_claims_graph(
    question
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
print("MODE")
print(result["mode"])

print("\nPLAN")
print(result["plan"])

print("\nANALYTICAL ROUTE")
print(result["analytical_route"])

print("\nKNOWLEDGE QUERY")
print(result["knowledge_query"])

print("\nSOURCES")
print(result["sources"])

print("\nANSWER")
print(result["answer"])

MODE
hybrid

PLAN
mode=<AgentMode.HYBRID: 'hybrid'> analytical_question='Compare persistent high-cost complex members with pharmacy-intensive chronic members.' knowledge_query='Definitions of Persistent High-Cost Complex and Pharmacy-Intensive Chronic.'

ANALYTICAL ROUTE
tool=<ToolName.SEGMENT_COMPARISON: 'compare_segments'> arguments={'segment_a': 'Persistent high-cost complex', 'segment_b': 'Pharmacy-intensive chronic'}

KNOWLEDGE QUERY
Definitions of Persistent High-Cost Complex and Pharmacy-Intensive Chronic.

SOURCES
['cohort_definitions.md', 'limitations.md']

ANSWER
**Cohort Definitions:**  
- **Persistent High-Cost Complex**: Defined by recurrent high-cost episodes and complex conditions (per project methodology in `cohort_definitions.md`).  
- **Pharmacy-Intensive Chronic**: Characterized by high prescription drug utilization, as inferred from synthetic CMS data (per `cohort_definitions.md`).  

**Key Utilization Differences:**  
1. **Annual Cost**: Persistent High-Cost Comple

In [2]:
# Checkign reporting layer

from agents.workflow import (
    ask_claims_agent,
)


response = ask_claims_agent(
    "Compare persistent high-cost complex members "
    "with pharmacy-intensive chronic members, "
    "explain how each cohort is defined, and summarize "
    "the main utilization differences."
)

response

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ClaimsAgentResponse(question='Compare persistent high-cost complex members with pharmacy-intensive chronic members, explain how each cohort is defined, and summarize the main utilization differences.', answer='**Cohort Definitions**  \n- **Persistent High-Cost Complex**: Defined in `cohort_definitions.md` as a segment characterized by recurrent high-cost utilization, chronic conditions, and inpatient services.  \n- **Pharmacy-Intensive Chronic**: Also from `cohort_definitions.md`, this cohort reflects members with elevated pharmacy utilization and chronic-condition management, but lower inpatient engagement.  \n\n**Key Utilization Differences**  \n1. **Cost and Inpatient Use**:  \n   - Persistent high-cost complex members had **$29,925.57 annual mean cost** (vs. $7,727.75 for pharmacy-intensive chronic members), driven by **71.96% inpatient prevalence** (vs. 0% in the other group) and **9.24 inpatient days** (vs. 0).  \n   - They also had higher outpatient claims (**8.60 vs. 5.13**) an

In [3]:
print(response.answer)

print(response.mode)

print(response.analytical_tool)

print(response.sources)

**Cohort Definitions**  
- **Persistent High-Cost Complex**: Defined in `cohort_definitions.md` as a segment characterized by recurrent high-cost utilization, chronic conditions, and inpatient services.  
- **Pharmacy-Intensive Chronic**: Also from `cohort_definitions.md`, this cohort reflects members with elevated pharmacy utilization and chronic-condition management, but lower inpatient engagement.  

**Key Utilization Differences**  
1. **Cost and Inpatient Use**:  
   - Persistent high-cost complex members had **$29,925.57 annual mean cost** (vs. $7,727.75 for pharmacy-intensive chronic members), driven by **71.96% inpatient prevalence** (vs. 0% in the other group) and **9.24 inpatient days** (vs. 0).  
   - They also had higher outpatient claims (**8.60 vs. 5.13**) and carrier claims (**47.75 vs. 29.40**).  

2. **Pharmacy Utilization**:  
   - Pharmacy-intensive chronic members exhibited more pharmacy events (**60.47 vs. 33.90**) and days supply (**2,046.26 vs. 1,081.54**), refle